# Leksjon 09 - Metakognisjon designmønster


## Oppsett

Denne notatblokken demonstrerer Metakognisjon designmønster ved bruk av Microsoft Agent Framework.

**Forutsetninger:**
- Azure OpenAI-distribusjon konfigurert via miljøvariabler
- Azure CLI autentisert (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Hva er metakognisjon?

Metakognisjon er **å tenke på tenkning**. I sammenheng med AI-agenter betyr det å bygge agenter som kan:

- **Selvreflektere** over sine egne resultater og resonnement
- **Oppdage feil** og komme seg elegant i stedet for å feile stille
- **Vurdere** om deres svar er komplette og hjelpsomme
- **Tilpasse** strategien sin når en innledende tilnærming ikke fungerer (f.eks. å falle tilbake på et reservedatasystem)

En metakognitiv agent svarer ikke bare på spørsmål — den overvåker egen ytelse og justerer underveis.


## Primær- og reserveverktøy

Et vanlig metakognisjonmønster er **fallback-strategien**. Agenten prøver et primærverktøy først; hvis det mislykkes (f.eks. en 404-feil), gjenkjenner agenten feilen og bytter transparent til et reserveverktøy.

Dette speiler virkelige systemer der primærtjenester kan være utilgjengelige og agenter må selvdiagnostisere problemet før de velger en alternativ vei.

Nedenfor definerer vi to verktøy for flysøk:
- **Primær** — dekker Paris, Tokyo og Barcelona
- **Reserve** — dekker Berlin, Sydney og New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Selvreflekterende agent med feilgjenoppretting

Agenten nedenfor er instruert til å prøve det primære flysystemet først, gjenkjenne feil og gjennomsiktig falle tilbake til backupsystemet. Etter hvert svar evaluerer den kort om den svarte fullt ut på brukerens spørsmål.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Mønster for egenvurdering

En annen side ved metakognisjon er **egenvurdering**: en separat agent (eller den samme agenten i et andre gjennomløp) vurderer et svar for fullstendighet, nøyaktighet og hjelpsomhet.

Nedenfor lager vi en `ResponseEvaluator`-agent som vurderer reiseagent-svar på tre dimensjoner.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sammendrag

I denne leksjonen lærte du hvordan du bygger **metakognitive agenter** ved hjelp av Microsoft Agent Framework:

- **Selvrefleksjon**: Agenter som overvåker egen resonnement og åpent kommuniserer hva som skjedde.
- **Feilgjenoppretting med fallback**: Et mønster med primært + backup-verktøy hvor agenten oppdager feil (f.eks. 404-feil) og automatisk prøver en alternativ kilde.
- **Selvvurdering**: En separat evaluator-agent som vurderer svar for fullstendighet, nøyaktighet og hjelpsomhet.

Disse mønstrene gjør agenter mer robuste, gjennomsiktige og pålitelige — kritiske egenskaper for produksjonsdistribusjoner.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vær oppmerksom på at automatiske oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på originalspråket skal betraktes som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
